In [ ]:
# Verificação de conectividade Supabase MCP
Este notebook verifica a conexão com o Supabase MCP e persiste o estado de progresso local.
</VSCode.Cell>
<VSCode.Cell language="python">
# 1. Instalar e importar bibliotecas
import os
import json
import requests
</VSCode.Cell>
<VSCode.Cell language="python">
# 2. Carregar estado anterior
state_file = 'supabase-progress-state.json'
if os.path.exists(state_file):
    with open(state_file, 'r', encoding='utf-8') as f:
        previous_state = json.load(f)
else:
    previous_state = {'last_step': None, 'connected': False}
print('Estado anterior:', previous_state)
</VSCode.Cell>
<VSCode.Cell language="python">
# 3. Configurar credenciais do Supabase MCP
env_path = '.env.local'
supabase_url = None
supabase_key = None
if os.path.exists(env_path):
    with open(env_path, 'r', encoding='utf-8') as f:
        for line in f.read().splitlines():
            if '=' not in line:
                continue
            k, v = line.split('=', 1)
            if k == 'NEXT_PUBLIC_SUPABASE_URL':
                supabase_url = v.strip()
            elif k == 'SUPABASE_SERVICE_ROLE_KEY' or k == 'SUPABASE_SERVICE_ROLE':
                supabase_key = v.strip()
print('Supabase URL encontrada:', bool(supabase_url))
print('Supabase key encontrada:', bool(supabase_key))
assert supabase_url and supabase_key, 'Credenciais Supabase MCP não encontradas em .env.local'
</VSCode.Cell>
<VSCode.Cell language="python">
# 4. Conectar ao Supabase
headers = {
    'apikey': supabase_key,
    'Authorization': f'Bearer {supabase_key}',
    'Content-Type': 'application/json'
}
try:
    response = requests.get(f'{supabase_url}/rest/v1/users?select=id&limit=1', headers=headers)
    response.raise_for_status()
    print('Cliente Supabase configurado com sucesso.')
except Exception as e:
    raise RuntimeError(f'Falha ao conectar ao Supabase MCP: {e}')
</VSCode.Cell>
<VSCode.Cell language="python">
# 5. Verificar conectividade
print('Status HTTP:', response.status_code)
print('Resposta:', response.text)
assert response.status_code == 200, 'A consulta de teste não retornou 200'
</VSCode.Cell>
<VSCode.Cell language="python">
# 6. Persistir progresso
new_state = {'last_step': 'connected_to_supabase_mcp', 'connected': True}
with open(state_file, 'w', encoding='utf-8') as f:
    json.dump(new_state, f, indent=2, ensure_ascii=False)
print('Progresso salvo em', state_file)
